# 구내식당 식수 인원 예측: 업그레이드 모델 (upmodel1)
## 분석 기반 업그레이드 포인트
1. **EDA 기반 기상 피처 고도화**: 단순 기온/강수량을 넘어 '불쾌지수(Discomfort Index)'와 '강수 여부'를 피처로 활용
2. **V5 모델링 기법 계승**: OOF(Out-of-Fold) 기반의 동적 가중치 앙상블 적용
3. **NLP 기반 메뉴 분석**: 단순 텍스트 파싱 대신 TF-IDF + SVD를 통한 메뉴 임베딩 적용
4. **정교한 인원 산출**: 본사정원수에서 휴가, 출장, 재택근무자를 제외한 '실질 출근 인원' 계산

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import KFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# 1. 데이터 로드
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')
weather = pd.read_csv('data/weather.csv')
submission = pd.read_csv('data/sample_submission.csv')

print("데이터 로드 완료")

In [ ]:
# 2. 피처 엔지니어링 (EDA 및 V5 인사이트 반영)
def preprocess_all(df, weather_df):
    df = df.copy()
    df['일자'] = pd.to_datetime(df['일자'])
    
    # 날짜 파생변수
    df['year'] = df['일자'].dt.year
    df['month'] = df['일자'].dt.month
    df['day'] = df['일자'].dt.day
    df['weekday'] = df['일자'].dt.weekday
    
    # 실질 출근 인원 (V5 핵심 피처)
    df['in_office'] = df['본사정원수'] - df['본사휴가자수'] - df['본사출장자수'] - df['현본사소속재택근무자수']
    
    # 기상 데이터 병합 및 불쾌지수 계산 (EDA 인사이트)
    w_df = weather_df.copy()
    w_df['일시'] = pd.to_datetime(w_df['일시'])
    w_df = w_df.groupby('일시').mean(numeric_only=True).reset_index()
    df = pd.merge(df, w_df[['일시', '기온', '강수량', '습도']], left_on='일자', right_on='일시', how='left')
    
    df['강수량'] = df['강수량'].fillna(0)
    df['is_rain'] = (df['강수량'] > 0).astype(int)
    
    # 불쾌지수 (DI = 0.81 * T + 0.01 * RH * (0.99 * T - 14.3) + 46.3)
    df['discomfort_index'] = 0.81 * df['기온'] + 0.01 * df['습도'] * (0.99 * df['기온'] - 14.3) + 46.3
    df['discomfort_index'] = df['discomfort_index'].fillna(df['discomfort_index'].mean())
    
    # 석식 0명 룰 피처 (자기계발의 날)
    df['is_culture_day'] = df['석식메뉴'].apply(lambda x: 1 if ('자기계발의날' in str(x) or '자기개발의날' in str(x)) else 0)
    
    return df

train_df = preprocess_all(train, weather)
test_df = preprocess_all(test, weather)

In [ ]:
# 3. NLP 메뉴 벡터화 (TF-IDF + SVD)
def extract_menu_vec(train_series, test_series, prefix, n_comp=5):
    vectorizer = TfidfVectorizer(max_features=500)
    combined = pd.concat([train_series, test_series]).fillna('')
    tfidf_matrix = vectorizer.fit_transform(combined)
    
    svd = TruncatedSVD(n_components=n_comp, random_state=42)
    svd_matrix = svd.fit_transform(tfidf_matrix)
    
    train_vec = svd_matrix[:len(train_series)]
    test_vec = svd_matrix[len(train_series):]
    
    train_cols = pd.DataFrame(train_vec, columns=[f'{prefix}_svd_{i}' for i in range(n_comp)])
    test_cols = pd.DataFrame(test_vec, columns=[f'{prefix}_svd_{i}' for i in range(n_comp)])
    
    return train_cols, test_cols

l_train_vec, l_test_vec = extract_menu_vec(train_df['중식메뉴'], test_df['중식메뉴'], 'lunch')
d_train_vec, d_test_vec = extract_menu_vec(train_df['석식메뉴'], test_df['석식메뉴'], 'dinner')

train_final = pd.concat([train_df.reset_index(drop=True), l_train_vec, d_train_vec], axis=1)
test_final = pd.concat([test_df.reset_index(drop=True), l_test_vec, d_test_vec], axis=1)

features = ['month', 'day', 'weekday', 'in_office', '본사시간외근무명령서승인건수', 'is_rain', 'discomfort_index', 'is_culture_day'] + \
           [f'lunch_svd_{i}' for i in range(5)] + [f'dinner_svd_{i}' for i in range(5)]

In [ ]:
# 4. OOF 가중치 최적화 앙상블 모델링
def train_ensemble(X, y, test_X, target_name):
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    
    models = {
        'XGB': XGBRegressor(n_estimators=1000, learning_rate=0.03, max_depth=6, random_state=42),
        'LGBM': LGBMRegressor(n_estimators=1000, learning_rate=0.03, max_depth=6, random_state=42, verbosity=-1),
        'CAT': CatBoostRegressor(n_estimators=1000, learning_rate=0.03, depth=6, random_state=42, silent=True)
    }
    
    oofs = {name: np.zeros(len(X)) for name in models.keys()}
    test_preds = {name: np.zeros(len(test_X)) for name in models.keys()}
    
    for name, model in models.items():
        print(f"Training {name} for {target_name}...")
        for fold, (t_idx, v_idx) in enumerate(kf.split(X, y)):
            X_train, y_train = X.iloc[t_idx], y.iloc[t_idx]
            X_val, y_val = X.iloc[v_idx], y.iloc[v_idx]
            
            model.fit(X_train, y_train)
            oofs[name][v_idx] = model.predict(X_val)
            test_preds[name] += model.predict(test_X) / 5
            
    # 가중치 산출 (MAE의 역수 비율)
    maes = {name: mean_absolute_error(y, oofs[name]) for name in models.keys()}
    inv_maes = {name: 1.0/maes[name] for name in models.keys()}
    weights = {name: inv_maes[name]/sum(inv_maes.values()) for name in models.keys()}
    
    final_pred = sum(test_preds[name] * weights[name] for name in models.keys())
    print(f"{target_name} Weights: {weights}")
    return final_pred

final_lunch = train_ensemble(train_final[features], train['중식계'], test_final[features], 'Lunch')
final_dinner = train_ensemble(train_final[features], train['석식계'], test_final[features], 'Dinner')

In [ ]:
# 5. 후처리 및 저장
# 석식 0명 처리 (is_culture_day)
final_dinner = np.where(test_final['is_culture_day'] == 1, 0, final_dinner)

submission['중식계'] = final_lunch
submission['석식계'] = final_dinner
submission.to_csv('upmodel1_submission.csv', index=False)
print("업그레이드 모델 제출 파일 'upmodel1_submission.csv' 저장 완료.")